# Brain Tumour Segmentation: Unified 3D Benchmark on BraTS 2020

One notebook trains and evaluates every 3D segmentation architecture of the benchmark (UNet, nnU-Net, VNet, UNETR, Swin UNETR, TransBTS) with identical preprocessing, split, training loop and evaluation. Each model takes the four MRI sequences of a patient (FLAIR, T1, T1CE, T2) as a 4-channel volume and predicts three nested regions: whole tumour (WT), tumour core (TC) and enhancing tumour (ET). It replaces the seven per-model notebooks.

**What changed relative to the earlier notebooks and the paper's stated limitations**

| Limitation or pending requirement | How this notebook handles it |
| --- | --- |
| Swin UNETR has no test result (its test cell failed) | Every model goes through the same evaluation code; Swin UNETR also gains the full-resolution skip of the reference design |
| Bootstrapped CIs and significance tests stated but not computed | Per-case metrics, bootstrap 95% CIs of every mean, pairwise Wilcoxon signed-rank tests with Holm correction |
| Dice averaged over batches, at the resampled 128³ grid, Dice only | Dice, HD95 (mm), sensitivity and specificity **per case**, at the **native** 240 x 240 x 155 resolution, with BraTS conventions for empty regions |
| 10 epochs insufficient for transformers; no warm-up | `EPOCHS` (default 50) for every model; linear warm-up then cosine decay for the transformer and hybrid models; training resumes across Kaggle sessions from per-epoch checkpoints |
| Single run, no multiple seeds | `SEEDS` list; results are averaged over seeds and seed-to-seed spread is reported |
| Split not stratified; claimed HGG/LGG balance unverified | Split stratified by tumour grade; results reported separately for HGG and LGG (bias check) |
| No ensembles | Seed ensembles per model, an all-model ensemble and a CNN-only ensemble |
| No uncertainty | Test-time augmentation (8 flips) gives both the prediction and a voxel-wise uncertainty, compared against per-case Dice |
| Inference speed and peak memory not benchmarked | Latency per volume (with and without TTA, mean ± sd), peak training and inference memory per model |
| Early stopping described as on validation loss but implemented on Dice | Model selection on validation mean Dice, stated as such; early stopping optional (`EARLY_STOP_PATIENCE`) |
| Deterministic algorithms claimed while `cudnn.benchmark` was on | `DETERMINISTIC` switch, reported in the run summary |
| One case skipped (misnamed mask of case 355) | Masks are found by name pattern, so every case with a mask is used |
| No ET post-processing; LGG cases without ET heavily penalised | Small ET predictions removed below a size threshold tuned on the validation cases |

What cannot be addressed in a notebook stays a stated limitation: no external multi-centre data (the BraTS validation set has no public labels), no large-scale pretraining, and a single input resolution per run (`IMG_SIZE` can be changed).

**Data.** Kaggle dataset `awsaf49/brats20-dataset-training-validation`, folder `BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData` (369 cases; grade from `name_mapping.csv`).

**Hardware.** Kaggle GPU T4 x2. Two worker processes train different (model, seed) jobs in parallel, one per GPU; volumes are preprocessed once into a memory-mapped cache.

**Outputs in `/kaggle/working/`**

```text
weights/<model>/<run>.pth   best-validation checkpoint of every run (<run>_last.pth while unfinished)
weights/deploy/VNet.pth     checkpoint for the web app + metadata.json
metrics/                    tables (CSV) and summaries (JSON); metrics/runs/<run>/ history, per-case test metrics, summary
predictions/<run>.npy       test probabilities at the model grid (uint8, 0-255) for ensembles and figures
plots/                      every figure as PNG
logs/                       notebook log, per-GPU worker logs and epoch events
src/                        the library and worker written by this notebook
outputs.zip                 everything above, built by the last cell
```

**Required Kaggle settings.** Accelerator **GPU T4 x2**; Internet only if `nibabel` is missing from the image; input dataset `awsaf49/brats20-dataset-training-validation`. The full benchmark (six models, `EPOCHS = 50`) does not fit in one 12-hour session: each session trains until `TRAINING_BUDGET_HOURS`, saves every unfinished run at the end of its last epoch, and still analyses whatever finished. To continue, attach this notebook's previous output as an input and set `RESUME_DIR`. To fit one session instead, reduce `EPOCHS` or `MODELS`.

**Runtime.** Not measured yet: the notebook has not been run in this form.

## 1. Setup

Imports; `nibabel` is installed only if missing. Expected output: library versions.

In [ ]:
import importlib
import subprocess
import sys

try:
    import nibabel as nib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nibabel"])
    import nibabel as nib

import json
import os
import shutil
import time
import zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from IPython.display import FileLink, display
from scipy.stats import spearmanr, wilcoxon
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print(f"torch {torch.__version__} | nibabel {nib.__version__} | numpy {np.__version__}", flush=True)

### Configuration

All settings in one place. Key choices:

- Recipes per model (optimiser, learning rate, warm-up, micro-batch, gradient checkpointing, deep-supervision weights) are in `volume_lib.RECIPES`; the paper's optimisers and learning rates are kept.
- `EFFECTIVE_BATCH = 4` for every model through gradient accumulation (Swin UNETR previously used 2).
- `EPOCHS` applies to every model, so convergence is no longer cut at ten epochs.
- `ET_CANDIDATES` are the enhancing-tumour size thresholds (voxels at native resolution) tried on the validation cases.

In [ ]:
DATASET_SLUG = "brats20-dataset-training-validation"
MODELS = ["UNet", "nnU-Net", "VNet", "UNETR", "Swin UNETR", "TransBTS"]
SEEDS = [42]
IMG_SIZE = 128
EPOCHS = 50
EFFECTIVE_BATCH = 4
GRAD_CLIP = 1.0
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15
SPLIT_SEED = 42
TTA = True
EARLY_STOP_PATIENCE = None  # e.g. 10 to stop after 10 epochs without a better validation Dice
ET_CANDIDATES = [0, 50, 100, 200, 300, 500, 800]
DETERMINISTIC = False
LOADER_WORKERS = 1  # per GPU process; Kaggle has four CPU cores for two workers and the notebook
LATENCY_ITERS = 20
BOOTSTRAP_SAMPLES = 2000
TRAINING_BUDGET_HOURS = 9.5
RESUME_DIR = None  # e.g. Path("/kaggle/input/<previous-version-output>") to continue a run
DEPLOY_MODEL = "VNet"  # architecture loaded by the web app

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SRC = WORK / "src"
PLOTS, METRICS, LOGS = WORK / "plots", WORK / "metrics", WORK / "logs"
WEIGHTS, PREDICTIONS = WORK / "weights", WORK / "predictions"
RUNS = METRICS / "runs"
CACHE = Path("/tmp/neuroscan-seg-cache")  # disposable preprocessed volumes, outside the outputs
CLAIMS = CACHE / "claims"
for folder in (SRC, PLOTS, METRICS, LOGS, WEIGHTS, PREDICTIONS, RUNS, CACHE, CLAIMS):
    folder.mkdir(parents=True, exist_ok=True)

REGIONS = ("WT", "TC", "ET")
MODEL_COLOURS = dict(zip(MODELS, plt.get_cmap("tab10").colors))
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "axes.grid": True, "grid.alpha": 0.3})
NOTEBOOK_START = time.time()


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(LOGS / "notebook.log", "a") as handle:
        handle.write(line + "\n")


def save_fig(fig, name):
    path = PLOTS / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    log(f"Saved plot {path.name}")


def save_table(frame, name, index=False, show=True):
    frame.to_csv(METRICS / f"{name}.csv", index=index)
    if show:
        display(frame)


log(f"Models: {MODELS} | seeds: {SEEDS} | epochs: {EPOCHS} | input {IMG_SIZE}^3")

### Hardware check

GPUs, CPU cores, RAM and free disk in the output folder and the cache, stopping unless two GPUs are visible. Expected output: two Tesla T4 lines.

In [ ]:
GPU_COUNT = torch.cuda.device_count()
for index in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(index)
    log(f"GPU {index}: {props.name}, {props.total_memory / 1024**3:.1f} GB")
memory = psutil.virtual_memory()
log(
    f"CPU cores: {os.cpu_count()} | RAM {memory.total / 1e9:.1f} GB total, {memory.available / 1e9:.1f} GB available"
)
log(
    f"Free disk: /kaggle/working {shutil.disk_usage(WORK).free / 1e9:.1f} GB | cache {shutil.disk_usage(CACHE).free / 1e9:.1f} GB"
)
assert (
    GPU_COUNT >= 2
), "Two GPUs are not visible: select 'GPU T4 x2' as the accelerator in the notebook settings."
DEVICE = torch.device("cuda:0")

### Shared library

Written to `src/volume_lib.py`: case discovery, foreground z-score normalisation and resampling, the cached dataset, GPU augmentation, the six architectures (VNet with the web app's exact module names), the Dice + BCE loss with deep supervision, flip test-time augmentation, the region hierarchy and ET post-processing, and per-case Dice, HD95, sensitivity and specificity.

In [ ]:
%%writefile /kaggle/working/src/volume_lib.py
"""Shared code for the unified BraTS 2020 segmentation notebook.

The notebook writes this file to /kaggle/working/src so the notebook and the training workers
(one process per GPU) import exactly the same data handling, models and metrics.
"""

from __future__ import annotations

import math
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

MODALITIES = ("flair", "t1", "t1ce", "t2")  # input channel order, as in the web app
REGIONS = ("WT", "TC", "ET")
HD95_EMPTY = 373.13  # BraTS value when exactly one of prediction and reference is empty

# Family, optimiser recipe and memory settings per model. CNNs keep the paper's SGD recipe;
# transformers keep AdamW and now get a linear warm-up before cosine decay.
RECIPES = {
    "UNet": {"family": "CNN", "optimizer": "sgd", "lr": 1e-2, "weight_decay": 3e-5, "warmup_epochs": 0, "micro_batch": 2, "checkpointing": False, "ds_weights": (1.0, 0.5, 0.25)},
    "nnU-Net": {"family": "CNN", "optimizer": "sgd", "lr": 1e-2, "weight_decay": 3e-5, "warmup_epochs": 0, "micro_batch": 2, "checkpointing": False, "ds_weights": (1.0, 0.5, 0.25, 0.125)},
    "VNet": {"family": "CNN", "optimizer": "sgd", "lr": 1e-2, "weight_decay": 3e-5, "warmup_epochs": 0, "micro_batch": 2, "checkpointing": False, "ds_weights": (1.0, 0.5, 0.25)},
    "UNETR": {"family": "Transformer", "optimizer": "adamw", "lr": 1e-4, "weight_decay": 1e-5, "warmup_epochs": 3, "micro_batch": 2, "checkpointing": True, "ds_weights": (1.0,)},
    "Swin UNETR": {"family": "Transformer", "optimizer": "adamw", "lr": 1e-4, "weight_decay": 1e-5, "warmup_epochs": 3, "micro_batch": 1, "checkpointing": True, "ds_weights": (1.0,)},
    "TransBTS": {"family": "Hybrid", "optimizer": "adamw", "lr": 1e-4, "weight_decay": 1e-5, "warmup_epochs": 3, "micro_batch": 2, "checkpointing": False, "ds_weights": (1.0, 0.5, 0.25)},
}


def slug(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def run_id(model, seed):
    return f"{slug(model)}_seed{seed}"


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------


def case_files(case_dir):
    """Modality and segmentation files of one BraTS case.

    The segmentation is any NIfTI whose name contains "seg", which also finds the misnamed
    mask of BraTS20_Training_355 that the earlier notebooks skipped.
    """
    case_dir = Path(case_dir)
    files = {}
    for path in sorted(case_dir.iterdir()):
        name = path.name.lower()
        if not name.endswith((".nii", ".nii.gz")):
            continue
        stem = name.split(".nii")[0]
        if "seg" in stem:
            files["seg"] = path
        else:
            for modality in MODALITIES:
                if stem.endswith("_" + modality):
                    files[modality] = path
    return files


def zscore_foreground(volume):
    brain = volume != 0
    if brain.any():
        volume[brain] = (volume[brain] - volume[brain].mean()) / (volume[brain].std() + 1e-8)
    return volume


def label_bits(seg):
    """BraTS labels -> bit field: 1 = WT (any tumour), 2 = TC (labels 1 and 4), 4 = ET (label 4)."""
    seg = np.asarray(seg)
    return ((seg > 0) * 1 + np.isin(seg, (1, 4)) * 2 + (seg == 4) * 4).astype(np.uint8)


def bits_to_regions(bits):
    return np.stack([(bits & 1) > 0, (bits & 2) > 0, (bits & 4) > 0])


def preprocess_case(args):
    """Load, normalise and resample one case (run in a worker process)."""
    import nibabel as nib

    torch.set_num_threads(1)
    files, size = args
    channels = []
    for modality in MODALITIES:
        volume = np.asarray(nib.load(str(files[modality])).dataobj, dtype=np.float32)
        volume = zscore_foreground(volume)
        resized = F.interpolate(torch.from_numpy(volume)[None, None], size=(size,) * 3, mode="trilinear", align_corners=False)
        channels.append(resized[0, 0].numpy().astype(np.float16))
    seg_image = nib.load(str(files["seg"]))
    seg = np.asarray(seg_image.dataobj, dtype=np.float32)
    resized_seg = F.interpolate(torch.from_numpy(seg)[None, None], size=(size,) * 3, mode="nearest")[0, 0].numpy()
    spacing = tuple(float(z) for z in seg_image.header.get_zooms()[:3])
    regions = bits_to_regions(label_bits(seg.astype(np.uint8)))
    voxel_cm3 = float(np.prod(spacing)) / 1000
    return {
        "image": np.stack(channels),
        "label": label_bits(resized_seg.astype(np.uint8)),
        "shape": seg.shape,
        "spacing": spacing,
        "volumes_cm3": [float(r.sum()) * voxel_cm3 for r in regions],
    }


def load_reference(seg_path):
    import nibabel as nib

    image = nib.load(str(seg_path))
    seg = np.asarray(image.dataobj).astype(np.uint8)
    return bits_to_regions(label_bits(seg)), tuple(float(z) for z in image.header.get_zooms()[:3])


class CachedCases(torch.utils.data.Dataset):
    """Cases from the memory-mapped cache; opened lazily so each loader worker maps the file."""

    def __init__(self, images_path, labels_path, indices):
        self.images_path, self.labels_path = images_path, labels_path
        self.indices = np.asarray(indices)
        self.images = self.labels = None

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        if self.images is None:
            self.images = np.load(self.images_path, mmap_mode="r")
            self.labels = np.load(self.labels_path, mmap_mode="r")
        index = self.indices[item]
        image = torch.from_numpy(np.asarray(self.images[index], dtype=np.float32))
        bits = torch.from_numpy(np.asarray(self.labels[index]))
        label = torch.stack([(bits & 1) > 0, (bits & 2) > 0, (bits & 4) > 0]).float()
        return image, label


def augment(image, label):
    """GPU augmentation of the earlier notebooks: random flips on each axis, per-channel
    intensity scale (0.9 to 1.1) and shift (-0.1 to 0.1)."""
    for axis in (2, 3, 4):
        flip = torch.rand(image.shape[0], device=image.device) < 0.5
        image = torch.where(flip[:, None, None, None, None], image.flip(axis), image)
        label = torch.where(flip[:, None, None, None, None], label.flip(axis), label)
    scale = torch.empty(image.shape[0], image.shape[1], 1, 1, 1, device=image.device).uniform_(0.9, 1.1)
    shift = torch.empty_like(scale).uniform_(-0.1, 0.1)
    return image * scale + shift, label


# ---------------------------------------------------------------------------
# Models (all from scratch in PyTorch; every model returns a tuple, main output first)
# ---------------------------------------------------------------------------


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet3D(nn.Module):
    """3D UNet, channels 32-64-128-256, max pooling, deep supervision at decoder stages 2 and 3."""

    def __init__(self):
        super().__init__()
        self.e1, self.e2, self.e3, self.b = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool = nn.MaxPool3d(2)
        self.u3, self.d3 = nn.ConvTranspose3d(256, 128, 2, 2), DoubleConv(256, 128)
        self.u2, self.d2 = nn.ConvTranspose3d(128, 64, 2, 2), DoubleConv(128, 64)
        self.u1, self.d1 = nn.ConvTranspose3d(64, 32, 2, 2), DoubleConv(64, 32)
        self.out, self.ds2, self.ds3 = nn.Conv3d(32, 3, 1), nn.Conv3d(64, 3, 1), nn.Conv3d(128, 3, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b = self.b(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1), self.ds2(d2), self.ds3(d3)


class NNUNet3D(nn.Module):
    """nnU-Net style full-resolution network: five stages 32-64-128-256-320, stride-2 convolutions,
    deep supervision at four decoder stages."""

    def __init__(self, f=(32, 64, 128, 256, 320)):
        super().__init__()
        self.e1, self.e2 = DoubleConv(4, f[0]), DoubleConv(f[0], f[1], 2)
        self.e3, self.e4, self.b = DoubleConv(f[1], f[2], 2), DoubleConv(f[2], f[3], 2), DoubleConv(f[3], f[4], 2)
        self.u4, self.d4 = nn.ConvTranspose3d(f[4], f[3], 2, 2), DoubleConv(2 * f[3], f[3])
        self.u3, self.d3 = nn.ConvTranspose3d(f[3], f[2], 2, 2), DoubleConv(2 * f[2], f[2])
        self.u2, self.d2 = nn.ConvTranspose3d(f[2], f[1], 2, 2), DoubleConv(2 * f[1], f[1])
        self.u1, self.d1 = nn.ConvTranspose3d(f[1], f[0], 2, 2), DoubleConv(2 * f[0], f[0])
        self.out = nn.Conv3d(f[0], 3, 1)
        self.ds2, self.ds3, self.ds4 = nn.Conv3d(f[1], 3, 1), nn.Conv3d(f[2], 3, 1), nn.Conv3d(f[3], 3, 1)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        b = self.b(s4)
        d4 = self.d4(torch.cat([self.u4(b), s4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), s3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), s2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), s1], 1))
        return self.out(d1), self.ds2(d2), self.ds3(d3), self.ds4(d4)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_convs):
        super().__init__()
        layers, ch = [], in_ch
        for _ in range(n_convs):
            layers += [nn.Conv3d(ch, out_ch, 3, padding=1, bias=False), nn.InstanceNorm3d(out_ch, affine=True), nn.PReLU(out_ch)]
            ch = out_ch
        self.convs = nn.Sequential(*layers)
        self.proj = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.convs(x) + self.proj(x)


class VNet3D(nn.Module):
    """Residual VNet with PReLU, channels 16-32-64-128.

    Module names match Backend/neuroscan/inference/networks.py, so the exported state dict loads
    into the web app unchanged.
    """

    def __init__(self):
        super().__init__()
        self.e1 = ResBlock(4, 16, 1)
        self.d1 = nn.Conv3d(16, 16, 2, stride=2, bias=False)
        self.e2 = ResBlock(16, 32, 1)
        self.d2 = nn.Conv3d(32, 32, 2, stride=2, bias=False)
        self.e3 = ResBlock(32, 64, 2)
        self.d3 = nn.Conv3d(64, 64, 2, stride=2, bias=False)
        self.b = ResBlock(64, 128, 2)
        self.u3 = nn.ConvTranspose3d(128, 64, 2, 2)
        self.r3 = ResBlock(128, 64, 2)
        self.u2 = nn.ConvTranspose3d(64, 32, 2, 2)
        self.r2 = ResBlock(64, 32, 1)
        self.u1 = nn.ConvTranspose3d(32, 16, 2, 2)
        self.r1 = ResBlock(32, 16, 1)
        self.out = nn.Conv3d(16, 3, 1)
        self.ds2 = nn.Conv3d(32, 3, 1)
        self.ds3 = nn.Conv3d(64, 3, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.d1(e1))
        e3 = self.e3(self.d2(e2))
        b = self.b(self.d3(e3))
        d3 = self.r3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.r2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.r1(torch.cat([self.u1(d2), e1], dim=1))
        return self.out(d1), self.ds2(d2), self.ds3(d3)


class TransformerLayer(nn.Module):
    """Pre-norm transformer layer."""

    def __init__(self, dim, heads, mlp_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(mlp_dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        y = self.norm1(x)
        x = x + self.attn(y, y, y, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))


class ConvBlock(nn.Module):
    """Two Conv-InstanceNorm-ReLU layers (UNETR decoder)."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNETR(nn.Module):
    """ViT-Base encoder (16^3 patches, 768 dim, 12 layers, 12 heads), skips from layers 3, 6, 9, 12."""

    def __init__(self, img_size=128, patch=16, dim=768, heads=12, layers=12, mlp_dim=3072, dropout=0.1):
        super().__init__()
        self.grid = img_size // patch
        self.patch_embed = nn.Conv3d(4, dim, patch, patch)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.grid**3, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.transformer = nn.ModuleList([TransformerLayer(dim, heads, mlp_dim, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
        self.use_checkpoint = False
        self.l4_main_up = nn.ConvTranspose3d(dim, 512, 2, 2)
        self.l4_skip_proj = ConvBlock(dim, 512)
        self.l4_skip_up = nn.ConvTranspose3d(512, 512, 2, 2)
        self.l4_conv = ConvBlock(1024, 256)
        self.l3_up = nn.ConvTranspose3d(256, 256, 2, 2)
        self.l3_skip_up = nn.Sequential(nn.ConvTranspose3d(dim, 256, 2, 2), nn.ConvTranspose3d(256, 256, 2, 2))
        self.l3_conv = ConvBlock(512, 128)
        self.l2_up = nn.ConvTranspose3d(128, 128, 2, 2)
        self.l2_skip_up = nn.Sequential(nn.ConvTranspose3d(dim, 256, 2, 2), nn.ConvTranspose3d(256, 128, 2, 2), nn.ConvTranspose3d(128, 128, 2, 2))
        self.l2_conv = ConvBlock(256, 64)
        self.l1_up = nn.ConvTranspose3d(64, 64, 2, 2)
        self.l1_skip_proj = ConvBlock(4, 64)
        self.l1_conv = ConvBlock(128, 32)
        self.out_conv = nn.Conv3d(32, 3, 1)

    def _volume(self, z):
        return z.transpose(1, 2).reshape(z.shape[0], z.shape[2], self.grid, self.grid, self.grid)

    def forward(self, x):
        tokens = self.pos_drop(self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos_embed)
        skips = {}
        for index, layer in enumerate(self.transformer, start=1):
            if self.use_checkpoint and self.training:
                tokens = checkpoint(layer, tokens, use_reentrant=False)
            else:
                tokens = layer(tokens)
            if index in (3, 6, 9, 12):
                skips[index] = tokens
        z3, z6, z9 = (self._volume(skips[i]) for i in (3, 6, 9))
        z12 = self._volume(self.norm(skips[12]))
        d4 = self.l4_conv(torch.cat([self.l4_main_up(z12), self.l4_skip_up(self.l4_skip_proj(z9))], 1))
        d3 = self.l3_conv(torch.cat([self.l3_up(d4), self.l3_skip_up(z6)], 1))
        d2 = self.l2_conv(torch.cat([self.l2_up(d3), self.l2_skip_up(z3)], 1))
        d1 = self.l1_conv(torch.cat([self.l1_up(d2), self.l1_skip_proj(x)], 1))
        return (self.out_conv(d1),)


def _window_partition(x, w):
    b, d, h, wd, c = x.shape
    x = x.view(b, d // w, w, h // w, w, wd // w, w, c).permute(0, 1, 3, 5, 2, 4, 6, 7)
    return x.reshape(-1, w * w * w, c)


def _window_reverse(windows, w, b, d, h, wd):
    x = windows.view(b, d // w, h // w, wd // w, w, w, w, -1).permute(0, 1, 4, 2, 5, 3, 6, 7)
    return x.reshape(b, d, h, wd, -1)


class WindowAttention3D(nn.Module):
    """Window self-attention with relative position bias, computed with scaled_dot_product_attention."""

    def __init__(self, dim, window, heads):
        super().__init__()
        self.heads, self.window = heads, window
        self.relative_position_bias_table = nn.Parameter(torch.zeros((2 * window - 1) ** 3, heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)
        coords = torch.stack(torch.meshgrid(*(torch.arange(window),) * 3, indexing="ij")).flatten(1)
        rel = (coords[:, :, None] - coords[:, None, :]).permute(1, 2, 0) + (window - 1)
        index = rel[..., 0] * (2 * window - 1) ** 2 + rel[..., 1] * (2 * window - 1) + rel[..., 2]
        self.register_buffer("relative_position_index", index, persistent=False)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, mask=None):
        windows, n, c = x.shape
        q, k, v = self.qkv(x).reshape(windows, n, 3, self.heads, c // self.heads).permute(2, 0, 3, 1, 4).unbind(0)
        bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(n, n, -1).permute(2, 0, 1)
        bias = bias.unsqueeze(0)  # (1, heads, n, n)
        if mask is not None:  # (nW, n, n): windows are ordered batch-major
            n_windows = mask.shape[0]
            bias = (bias.unsqueeze(0) + mask[None, :, None]).expand(windows // n_windows, -1, -1, -1, -1)
            bias = bias.reshape(windows, self.heads, n, n)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.proj(out.transpose(1, 2).reshape(windows, n, c))


class SwinBlock3D(nn.Module):
    def __init__(self, dim, resolution, heads, window, shift):
        super().__init__()
        self.resolution, self.window, self.shift = resolution, window, shift
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention3D(dim, window, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.padded = tuple(int(math.ceil(r / window)) * window for r in resolution)
        mask = None
        if shift:
            regions = torch.zeros(1, *self.padded, 1)
            spans = (slice(0, -window), slice(-window, -shift), slice(-shift, None))
            label = 0
            for sd in spans:
                for sh in spans:
                    for sw in spans:
                        regions[:, sd, sh, sw, :] = label
                        label += 1
            ids = _window_partition(regions, window)[..., 0]
            mask = (ids[:, None, :] - ids[:, :, None]).ne(0).float() * -100.0
        self.register_buffer("attn_mask", mask, persistent=False)

    def forward(self, x):
        b, length, c = x.shape
        d, h, w = self.resolution
        dp, hp, wp = self.padded
        shortcut = x
        x = F.pad(self.norm1(x).view(b, d, h, w, c), (0, 0, 0, wp - w, 0, hp - h, 0, dp - d))
        if self.shift:
            x = torch.roll(x, shifts=(-self.shift,) * 3, dims=(1, 2, 3))
        x = _window_reverse(self.attn(_window_partition(x, self.window), self.attn_mask), self.window, b, dp, hp, wp)
        if self.shift:
            x = torch.roll(x, shifts=(self.shift,) * 3, dims=(1, 2, 3))
        x = shortcut + x[:, :d, :h, :w, :].reshape(b, length, c)
        return x + self.mlp(self.norm2(x))


class PatchMerging3D(nn.Module):
    def __init__(self, resolution, dim):
        super().__init__()
        self.resolution = resolution
        self.norm = nn.LayerNorm(8 * dim)
        self.reduce = nn.Linear(8 * dim, 2 * dim, bias=False)

    def forward(self, x):
        d, h, w = self.resolution
        x = x.view(x.shape[0], d, h, w, -1)
        x = torch.cat([x[:, i::2, j::2, k::2] for i in range(2) for j in range(2) for k in range(2)], -1)
        return self.reduce(self.norm(x.reshape(x.shape[0], -1, x.shape[-1])))


class SwinUNETR(nn.Module):
    """Swin UNETR: 2^3 patches, embed 48 doubling per stage, window 7, depths 2, heads 3-6-12-24.

    Compared with the earlier notebook it adds the full-resolution convolutional skip of the
    reference design, so the last decoder stage no longer upsamples without any skip.
    """

    def __init__(self, img_size=128, embed=48, window=7, depths=(2, 2, 2, 2), heads=(3, 6, 12, 24)):
        super().__init__()
        tokens = img_size // 2
        dims = [embed * 2**i for i in range(4)]
        self.resolutions = [(tokens // 2**i,) * 3 for i in range(4)]
        self.patch_embed = nn.Conv3d(4, embed, 2, 2)
        self.embed_norm = nn.LayerNorm(embed)
        self.stages = nn.ModuleList()
        self.merges = nn.ModuleList()
        self.skip_norms = nn.ModuleList()
        for i in range(4):
            self.stages.append(nn.ModuleList([SwinBlock3D(dims[i], self.resolutions[i], heads[i], window, 0 if j % 2 == 0 else window // 2) for j in range(depths[i])]))
            self.merges.append(PatchMerging3D(self.resolutions[i], dims[i]) if i < 3 else nn.Identity())
            self.skip_norms.append(nn.LayerNorm(dims[i]))
        self.use_checkpoint = False
        self.e0 = DoubleConv(4, 32)
        self.up3, self.dc3 = nn.ConvTranspose3d(dims[3], dims[2], 2, 2), DoubleConv(2 * dims[2], dims[2])
        self.up2, self.dc2 = nn.ConvTranspose3d(dims[2], dims[1], 2, 2), DoubleConv(2 * dims[1], dims[1])
        self.up1, self.dc1 = nn.ConvTranspose3d(dims[1], dims[0], 2, 2), DoubleConv(2 * dims[0], dims[0])
        self.up0, self.dc0 = nn.ConvTranspose3d(dims[0], 32, 2, 2), DoubleConv(64, 32)
        self.out = nn.Conv3d(32, 3, 1)

    def forward(self, x):
        t = self.embed_norm(self.patch_embed(x).flatten(2).transpose(1, 2))
        skips = []
        for i, (blocks, merge) in enumerate(zip(self.stages, self.merges)):
            for block in blocks:
                t = checkpoint(block, t, use_reentrant=False) if self.use_checkpoint and self.training else block(t)
            d = self.resolutions[i][0]
            skips.append(self.skip_norms[i](t).transpose(1, 2).reshape(t.shape[0], -1, d, d, d))
            t = merge(t)
        d3 = self.dc3(torch.cat([self.up3(skips[3]), skips[2]], 1))
        d2 = self.dc2(torch.cat([self.up2(d3), skips[1]], 1))
        d1 = self.dc1(torch.cat([self.up1(d2), skips[0]], 1))
        d0 = self.dc0(torch.cat([self.up0(d1), self.e0(x)], 1))
        return (self.out(d0),)


class ResConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.shortcut = (
            nn.Sequential(nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False), nn.InstanceNorm3d(out_ch, affine=True))
            if in_ch != out_ch or stride != 1
            else nn.Identity()
        )

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, 2)
        self.conv = ResConvBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], 1))


class TransBTS(nn.Module):
    """Residual CNN encoder 32-64-128-256-512, four-layer transformer bottleneck (8 heads, 512 dim)."""

    def __init__(self, img_size=128, dim=512, depth=4, heads=8):
        super().__init__()
        self.e1, self.e2 = ResConvBlock(4, 32), ResConvBlock(32, 64, 2)
        self.e3, self.e4 = ResConvBlock(64, 128, 2), ResConvBlock(128, 256, 2)
        self.b = ResConvBlock(256, dim, 2)
        self.pos_embed = nn.Parameter(torch.zeros(1, (img_size // 16) ** 3, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.transformer = nn.Sequential(*[TransformerLayer(dim, heads, 4 * dim) for _ in range(depth)])
        self.trans_norm = nn.LayerNorm(dim)
        self.d4, self.d3 = UpBlock(dim, 256, 256), UpBlock(256, 128, 128)
        self.d2, self.d1 = UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.out, self.ds2, self.ds3 = nn.Conv3d(32, 3, 1), nn.Conv3d(64, 3, 1), nn.Conv3d(128, 3, 1)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        b = self.b(s4)
        n, c, d, h, w = b.shape
        tokens = self.trans_norm(self.transformer(b.flatten(2).transpose(1, 2) + self.pos_embed))
        b = tokens.transpose(1, 2).reshape(n, c, d, h, w)
        d3 = self.d3(self.d4(b, s4), s3)
        d2 = self.d2(d3, s2)
        d1 = self.d1(d2, s1)
        return self.out(d1), self.ds2(d2), self.ds3(d3)


def build_model(name, img_size):
    builders = {
        "UNet": UNet3D,
        "nnU-Net": NNUNet3D,
        "VNet": VNet3D,
        "UNETR": lambda: UNETR(img_size),
        "Swin UNETR": lambda: SwinUNETR(img_size),
        "TransBTS": lambda: TransBTS(img_size),
    }
    model = builders[name]()
    if RECIPES[name]["checkpointing"]:
        model.use_checkpoint = True
    return model


# ---------------------------------------------------------------------------
# Loss, inference, post-processing, metrics
# ---------------------------------------------------------------------------


def region_loss(logits, target):
    """Soft Dice over the batch plus binary cross-entropy, per region, as in the paper."""
    logits = logits.float()
    prob = torch.sigmoid(logits)
    dims = (0, 2, 3, 4)
    dice = 1 - (2 * (prob * target).sum(dims) + 1e-6) / (prob.sum(dims) + target.sum(dims) + 1e-6)
    return dice.mean() + F.binary_cross_entropy_with_logits(logits, target)


def supervised_loss(outputs, target, weights):
    loss = 0.0
    for output, weight in zip(outputs, weights):
        scaled = target if output.shape[2:] == target.shape[2:] else F.interpolate(target, size=output.shape[2:], mode="nearest")
        loss = loss + weight * region_loss(output, scaled)
    return loss


FLIP_SETS = [(), (2,), (3,), (4,), (2, 3), (2, 4), (3, 4), (2, 3, 4)]


@torch.no_grad()
def predict_probs(model, x, tta):
    """Sigmoid probabilities of the main output, averaged over the 8 axis flips when tta is on.

    Returns the mean and the standard deviation across flips (zero without TTA).
    """
    model.eval()
    flips = FLIP_SETS if tta else [()]
    total = total_sq = None
    for dims in flips:
        inputs = x.flip(dims) if dims else x
        with torch.autocast("cuda", dtype=torch.float16):
            prob = torch.sigmoid(model(inputs)[0].float())
        prob = prob.flip(dims) if dims else prob
        total = prob if total is None else total + prob
        total_sq = prob**2 if total_sq is None else total_sq + prob**2
    mean = total / len(flips)
    std = (total_sq / len(flips) - mean**2).clamp_min(0).sqrt()
    return mean, std


def to_native(probs, shape):
    """Resample (3, S, S, S) probabilities to the original grid (trilinear on probabilities)."""
    return F.interpolate(probs[None].float(), size=tuple(shape), mode="trilinear", align_corners=False)[0]


def regions_from_probs(probs, et_min_voxels=0, threshold=0.5):
    """Nested regions with the BraTS label hierarchy: ET inside TC inside WT.

    An ET prediction smaller than et_min_voxels is removed, a standard BraTS post-processing
    step that helps cases without enhancing tumour (mostly lower-grade gliomas).
    """
    wt, tc, et = (probs > threshold).unbind(0)
    if int(et.sum()) < et_min_voxels:
        et = torch.zeros_like(et)
    tc = tc | et
    wt = wt | tc
    return torch.stack([wt, tc, et])


def dice(pred, ref):
    pred_sum, ref_sum = int(pred.sum()), int(ref.sum())
    if pred_sum + ref_sum == 0:
        return 1.0
    return 2 * int((pred & ref).sum()) / (pred_sum + ref_sum)


def hd95(pred, ref, spacing):
    """95th-percentile symmetric Hausdorff distance in mm (MedPy definition), on the bounding box
    of both masks for speed."""
    from scipy import ndimage

    if not pred.any() and not ref.any():
        return 0.0
    if not pred.any() or not ref.any():
        return HD95_EMPTY
    coords = np.argwhere(pred | ref)
    low = np.maximum(coords.min(0) - 2, 0)
    high = coords.max(0) + 3
    box = tuple(slice(a, b) for a, b in zip(low, high))
    pred = np.pad(pred[box], 2)
    ref = np.pad(ref[box], 2)
    structure = ndimage.generate_binary_structure(3, 1)
    pred_surface = pred ^ ndimage.binary_erosion(pred, structure)
    ref_surface = ref ^ ndimage.binary_erosion(ref, structure)
    to_ref = ndimage.distance_transform_edt(~ref_surface, sampling=spacing)[pred_surface]
    to_pred = ndimage.distance_transform_edt(~pred_surface, sampling=spacing)[ref_surface]
    return float(max(np.percentile(to_ref, 95), np.percentile(to_pred, 95)))


def case_metrics(pred, ref, spacing):
    """Dice, HD95, sensitivity, specificity and volumes for the three regions of one case."""
    voxel_cm3 = float(np.prod(spacing)) / 1000
    row = {}
    for index, region in enumerate(REGIONS):
        p, r = pred[index], ref[index]
        tp = int((p & r).sum())
        fn = int(r.sum()) - tp
        fp = int(p.sum()) - tp
        tn = p.size - tp - fn - fp
        row[f"dice_{region}"] = dice(p, r)
        row[f"hd95_{region}"] = hd95(p, r, spacing)
        row[f"sensitivity_{region}"] = tp / (tp + fn) if tp + fn else 1.0
        row[f"specificity_{region}"] = tn / (tn + fp) if tn + fp else 1.0
        row[f"volume_pred_cm3_{region}"] = float(p.sum()) * voxel_cm3
        row[f"volume_ref_cm3_{region}"] = float(r.sum()) * voxel_cm3
    row["dice_mean"] = float(np.mean([row[f"dice_{r}"] for r in REGIONS]))
    row["hd95_mean"] = float(np.mean([row[f"hd95_{r}"] for r in REGIONS]))
    return row


def bootstrap_mean_ci(values, n_boot=2000, seed=0):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n_boot, len(values)))].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

### Training worker

Written to `src/volume_worker.py` and launched once per GPU. Per job it probes the largest micro-batch that fits, trains with fp16 autocast, gradient scaling, accumulation to `EFFECTIVE_BATCH`, gradient clipping and a per-step learning-rate schedule, validates every epoch, keeps the best-Dice checkpoint and a resumable last checkpoint, tunes the ET size threshold on validation, then evaluates every test case at native resolution with TTA and times inference.

In [ ]:
%%writefile /kaggle/working/src/volume_worker.py
"""Training worker for the unified segmentation notebook: one process per GPU.

Each worker claims (model, seed) jobs, trains with checkpoints every epoch (so a later session
resumes from the last epoch), selects the best epoch on validation mean Dice, tunes the
enhancing-tumour size threshold on the validation cases, and evaluates the test cases at their
native resolution with test-time augmentation.
"""

import argparse
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent))
import shared_lib  # noqa: E402
import volume_lib as lib  # noqa: E402


def seed_everything(seed, deterministic):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)


def probe_micro_batch(model, start, size, weights, device):
    """Largest micro-batch whose forward and backward pass fit; nothing is stepped."""
    model.train()
    candidate = start
    while candidate >= 1:
        try:
            x = torch.randn(candidate, 4, size, size, size, device=device)
            y = torch.zeros(candidate, 3, size, size, size, device=device)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lib.supervised_loss(model(x), y, weights)
            loss.backward()
            torch.cuda.synchronize(device)
            return candidate
        except torch.OutOfMemoryError:
            candidate //= 2
        finally:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    raise RuntimeError("Even a micro-batch of 1 does not fit in GPU memory")


@torch.no_grad()
def validate(model, loader, weights, device):
    model.eval()
    losses, rows = [], []
    for image, label in loader:
        image, label = image.to(device, non_blocking=True), label.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(image)
        losses.append(lib.supervised_loss(outputs, label, weights).item())
        pred = torch.sigmoid(outputs[0].float()) > 0.5
        for p, r in zip(pred, label.bool()):
            rows.append([lib.dice(p[i], r[i]) for i in range(3)])
    rows = np.array(rows)
    return float(np.mean(losses)), rows.mean(axis=0)


def evaluate_split(model, cases, cfg, device, tta, et_candidates=None, et_min=0, probs_out=None):
    """Native-resolution evaluation of a set of cases.

    With et_candidates, returns the mean ET Dice for each candidate size threshold (tuning).
    Otherwise returns one metrics row per case and optionally stores the probabilities.
    """
    images = np.load(cfg["images_path"], mmap_mode="r")
    tuning = {threshold: [] for threshold in (et_candidates or [])}
    rows = []
    for position, case in enumerate(cases.itertuples()):
        x = torch.from_numpy(np.asarray(images[case.cache_index], dtype=np.float32))[None].to(device)
        mean, std = lib.predict_probs(model, x, tta)
        mean, std = mean[0], std[0]
        reference, spacing = lib.load_reference(case.seg)
        native = lib.to_native(mean, reference.shape[1:])
        if et_candidates is not None:
            ref_et = torch.from_numpy(reference[2]).to(device)
            for threshold in et_candidates:
                tuning[threshold].append(lib.dice(lib.regions_from_probs(native, threshold)[2], ref_et))
            continue
        pred = lib.regions_from_probs(native, et_min).cpu().numpy()
        wt_pred = mean[0] > 0.5
        uncertainty = float(std[0][wt_pred].mean()) if wt_pred.any() else 0.0
        rows.append({"case": case.case, "grade": case.grade, **lib.case_metrics(pred, reference, spacing), "tta_std_in_wt": uncertainty})
        if probs_out is not None:
            probs_out[position] = (mean.clamp(0, 1) * 255).round().to(torch.uint8).cpu().numpy()
    if et_candidates is not None:
        return {threshold: float(np.mean(values)) for threshold, values in tuning.items()}
    return pd.DataFrame(rows)


@torch.no_grad()
def time_inference(model, size, device, tta, iterations, warmup):
    x = torch.randn(1, 4, size, size, size, device=device)
    for _ in range(warmup):
        lib.predict_probs(model, x, tta)
    torch.cuda.synchronize(device)
    timings = []
    for _ in range(iterations):
        start = time.perf_counter()
        lib.predict_probs(model, x, tta)
        torch.cuda.synchronize(device)
        timings.append(time.perf_counter() - start)
    return float(np.mean(timings)), float(np.std(timings))


def run_job(job, cfg, cases, device, reporter):
    name, seed = job["model"], job["seed"]
    rid = lib.run_id(name, seed)
    recipe = lib.RECIPES[name]
    size, epochs = cfg["img_size"], cfg["epochs"]
    seed_everything(seed, cfg["deterministic"])
    run_dir = Path(cfg["runs_dir"]) / rid
    run_dir.mkdir(parents=True, exist_ok=True)
    weights_dir = Path(cfg["weights_dir"]) / lib.slug(name)
    weights_dir.mkdir(parents=True, exist_ok=True)
    best_path, last_path = weights_dir / f"{rid}.pth", weights_dir / f"{rid}_last.pth"

    model = lib.build_model(name, size).to(device)
    weights = recipe["ds_weights"]
    micro = probe_micro_batch(model, recipe["micro_batch"], size, weights, device)
    accumulation = math.ceil(cfg["effective_batch"] / micro)
    train = cases[cases["role"] == "train"]
    val = cases[cases["role"] == "val"]
    loader_kwargs = {"num_workers": cfg["loader_workers"], "pin_memory": True, "persistent_workers": cfg["loader_workers"] > 0}
    train_loader = torch.utils.data.DataLoader(
        lib.CachedCases(cfg["images_path"], cfg["labels_path"], train["cache_index"].to_numpy()),
        batch_size=micro,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        **loader_kwargs,
    )
    val_loader = torch.utils.data.DataLoader(
        lib.CachedCases(cfg["images_path"], cfg["labels_path"], val["cache_index"].to_numpy()), batch_size=micro, **loader_kwargs
    )
    steps_per_epoch = math.ceil(len(train_loader) / accumulation)
    total_steps = epochs * steps_per_epoch
    warmup_steps = recipe["warmup_epochs"] * steps_per_epoch
    if recipe["optimizer"] == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=recipe["lr"], momentum=0.99, nesterov=True, weight_decay=recipe["weight_decay"])

        def lr_factor(step):
            return (1 - min(step, total_steps - 1) / total_steps) ** 0.9

    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=recipe["lr"], weight_decay=recipe["weight_decay"])

        def lr_factor(step):
            if step < warmup_steps:
                return (step + 1) / warmup_steps
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda")
    state = {"epoch": 0, "step": 0, "history": [], "best_dice": -1.0, "best_epoch": 0, "stale": 0, "train_seconds": 0.0, "peak_memory_gb": 0.0}
    if last_path.exists():  # resume from the end of the last finished epoch
        checkpoint = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        state = checkpoint["state"]
        torch.set_rng_state(checkpoint["cpu_rng"].cpu())
        reporter.event(f"{rid}: resumed after epoch {state['epoch']}")
    reporter.event(
        f"{rid}: {len(train)} train / {len(val)} val cases, micro-batch {micro} x {accumulation} accumulation, "
        f"{total_steps} steps over {epochs} epochs"
    )

    torch.cuda.reset_peak_memory_stats(device)
    while state["epoch"] < epochs:
        if time.time() > cfg["deadline"]:
            reporter.event(f"{rid}: time budget reached after epoch {state['epoch']}; resume in the next session")
            return False
        epoch = state["epoch"] + 1
        model.train()
        start, losses = time.perf_counter(), []
        optimizer.zero_grad(set_to_none=True)
        for batch_index, (image, label) in enumerate(train_loader, start=1):
            image, label = lib.augment(image.to(device, non_blocking=True), label.to(device, non_blocking=True))
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(image)
            loss = lib.supervised_loss(outputs, label, weights)
            scaler.scale(loss / accumulation).backward()
            losses.append(loss.item())
            if batch_index % accumulation == 0 or batch_index == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                state["step"] += 1
                reporter.status(
                    job=rid,
                    step=state["step"],
                    total=total_steps,
                    postfix={"epoch": epoch, "loss": round(float(np.mean(losses)), 4), "lr": f"{scheduler.get_last_lr()[0]:.2e}", "mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2)},
                )
        train_seconds = time.perf_counter() - start
        val_loss, val_dice = validate(model, val_loader, weights, device)
        state["train_seconds"] += train_seconds
        state["peak_memory_gb"] = max(state["peak_memory_gb"], torch.cuda.max_memory_allocated(device) / 1e9)
        row = {
            "epoch": epoch,
            "lr": scheduler.get_last_lr()[0],
            "train_loss": float(np.mean(losses)),
            "val_loss": val_loss,
            **{f"val_dice_{region}": float(value) for region, value in zip(lib.REGIONS, val_dice)},
            "val_dice_mean": float(val_dice.mean()),
            "epoch_seconds": time.perf_counter() - start,
        }
        state["history"].append(row)
        improved = row["val_dice_mean"] > state["best_dice"]
        if improved:
            state["best_dice"], state["best_epoch"], state["stale"] = row["val_dice_mean"], epoch, 0
            torch.save({k: v.detach().float().cpu() for k, v in model.state_dict().items()}, best_path)
        else:
            state["stale"] += 1
        state["epoch"] = epoch
        pd.DataFrame(state["history"]).to_csv(run_dir / "history.csv", index=False)
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "state": state, "cpu_rng": torch.get_rng_state()},
            last_path,
        )
        reporter.event(
            f"{rid} epoch {epoch}/{epochs}: train loss {row['train_loss']:.4f} | val loss {val_loss:.4f} | "
            f"val Dice WT {val_dice[0]:.4f} TC {val_dice[1]:.4f} ET {val_dice[2]:.4f} mean {row['val_dice_mean']:.4f}"
            + (" | best" if improved else "")
        )
        if cfg["early_stop_patience"] and state["stale"] >= cfg["early_stop_patience"]:
            reporter.event(f"{rid}: early stop after {state['stale']} epochs without improvement")
            break

    # Evaluation of the best epoch
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    model.eval()
    torch.cuda.reset_peak_memory_stats(device)
    tuning = evaluate_split(model, val, cfg, device, cfg["tta"], et_candidates=cfg["et_candidates"])
    et_min = max(tuning, key=lambda t: (tuning[t], -t))
    test = cases[cases["role"] == "test"]
    probs_path = Path(cfg["probs_dir"]) / f"{rid}.npy"
    probs_out = np.lib.format.open_memmap(probs_path, mode="w+", dtype=np.uint8, shape=(len(test), 3, size, size, size))
    metrics = evaluate_split(model, test, cfg, device, cfg["tta"], et_min=et_min, probs_out=probs_out)
    probs_out.flush()
    del probs_out
    metrics.to_csv(run_dir / "test_cases.csv", index=False)
    inference_memory = torch.cuda.max_memory_allocated(device) / 1e9
    latency, latency_sd = time_inference(model, size, device, False, cfg["latency_iters"], 5)
    latency_tta, latency_tta_sd = time_inference(model, size, device, True, max(3, cfg["latency_iters"] // 5), 1)
    summary = {
        "run_id": rid,
        "model": name,
        "family": recipe["family"],
        "seed": seed,
        "parameters": sum(p.numel() for p in model.parameters()),
        "micro_batch": micro,
        "accumulation": accumulation,
        "optimizer_steps": state["step"],
        "epochs_run": state["epoch"],
        "best_epoch": state["best_epoch"],
        "best_val_dice": state["best_dice"],
        "et_min_voxels": et_min,
        "et_tuning": {str(k): v for k, v in tuning.items()},
        "train_seconds": state["train_seconds"],
        "peak_train_memory_gb": state["peak_memory_gb"],
        "peak_inference_memory_gb": inference_memory,
        "latency_s": latency,
        "latency_s_sd": latency_sd,
        "latency_tta_s": latency_tta,
        "latency_tta_s_sd": latency_tta_sd,
        "test_dice_mean": float(metrics["dice_mean"].mean()),
        "gpu": torch.cuda.get_device_name(device),
    }
    shared_lib.write_json(run_dir / "summary.json", summary)
    last_path.unlink(missing_ok=True)  # optimiser state is no longer needed
    reporter.event(f"{rid} done: test mean Dice {summary['test_dice_mean']:.4f}, ET size threshold {et_min} voxels")
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    return True


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--gpu", type=int, required=True)
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    device = torch.device("cuda:0")  # CUDA_VISIBLE_DEVICES selects the physical GPU
    torch.cuda.set_device(device)
    reporter = shared_lib.Reporter(cfg["log_dir"], args.gpu)
    cases = pd.read_csv(cfg["case_table"])
    claims = Path(cfg["claims_dir"])
    for job in cfg["jobs"]:
        rid = lib.run_id(job["model"], job["seed"])
        if (Path(cfg["runs_dir"]) / rid / "summary.json").exists():
            continue
        if time.time() > cfg["deadline"]:
            reporter.event(f"Time budget reached; {rid} and later jobs left for the next session")
            break
        try:
            os.close(os.open(claims / rid, os.O_CREAT | os.O_EXCL | os.O_WRONLY))
        except FileExistsError:
            continue
        run_job(job, cfg, cases, device, reporter)
    reporter.status(job=None)


if __name__ == "__main__":
    main()

### Worker launcher, monitor and shared helpers

Written to `src/shared_lib.py`: starts one worker per GPU (each sees one device through `CUDA_VISIBLE_DEVICES`), polls their status files to draw an overall jobs bar, one step bar per GPU and a line per finished epoch, and stops the cell with the end of a worker's log if it fails. Also holds the workers' progress reporter and the Holm correction.

In [ ]:
%%writefile /kaggle/working/src/shared_lib.py
"""Helpers shared by the unified notebooks: worker launch and live monitoring, progress
reporting from the workers, JSON files, and the Holm correction.

Each notebook writes this file to /kaggle/working/src; training runs in one subprocess per GPU
and the notebook turns the workers' status files into live progress bars.
"""

import json
import os
import sys
import time
from pathlib import Path

import numpy as np


def read_json(path, default=None):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return default


def write_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, default=float))
    os.replace(tmp, path)


def launch_workers(script, config_path, log_dir, gpu_count):
    import subprocess

    processes = []
    for gpu in range(gpu_count):
        env = os.environ.copy()
        env.update(
            CUDA_VISIBLE_DEVICES=str(gpu),
            PYTHONUNBUFFERED="1",
            CUBLAS_WORKSPACE_CONFIG=":4096:8",
            OMP_NUM_THREADS=str(max(1, (os.cpu_count() or 2) // gpu_count)),
        )
        log = open(Path(log_dir) / f"worker_gpu{gpu}.log", "a")
        command = [sys.executable, str(script), "--config", str(config_path), "--gpu", str(gpu)]
        processes.append((subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    return processes


def monitor_workers(processes, log_dir, total_jobs, finished_jobs, poll_seconds=3):
    """Live progress: one bar for all jobs, one bar per GPU, epoch lines as they arrive."""
    from tqdm.auto import tqdm

    log_dir = Path(log_dir)
    overall = tqdm(total=total_jobs, desc="Jobs", unit="job", position=0)
    overall.update(finished_jobs())
    bars = [
        tqdm(total=1, desc=f"GPU {gpu}: waiting", unit="step", position=gpu + 1, leave=True)
        for gpu in range(len(processes))
    ]
    offsets = [0] * len(processes)
    current_job = [None] * len(processes)
    while True:
        for gpu, bar in enumerate(bars):
            status = read_json(log_dir / f"status_gpu{gpu}.json", {})
            if status.get("job"):
                if status["job"] != current_job[gpu]:
                    current_job[gpu] = status["job"]
                    bar.reset(total=max(1, status.get("total", 1)))
                    bar.set_description(f"GPU {gpu}: {status['job']}")
                bar.n = min(status.get("step", 0), bar.total)
                bar.set_postfix(status.get("postfix", {}), refresh=False)
                bar.refresh()
            events = log_dir / f"events_gpu{gpu}.jsonl"
            if events.exists():
                with open(events) as handle:
                    handle.seek(offsets[gpu])
                    for line in handle:
                        tqdm.write(json.loads(line)["message"])
                    offsets[gpu] = handle.tell()
        done = finished_jobs()
        overall.update(done - overall.n)
        if all(process.poll() is not None for process, _ in processes):
            break
        time.sleep(poll_seconds)
    for bar in bars:
        bar.close()
    overall.close()
    failures = []
    for gpu, (process, log) in enumerate(processes):
        log.close()
        if process.returncode != 0:
            tail = (log_dir / f"worker_gpu{gpu}.log").read_text().splitlines()[-40:]
            failures.append(f"GPU {gpu} worker exited with {process.returncode}:\n" + "\n".join(tail))
    if failures:
        raise RuntimeError("\n\n".join(failures))


class Reporter:
    """Status file (polled by the notebook) and epoch events written by a worker."""

    def __init__(self, log_dir, gpu):
        self.status_path = Path(log_dir) / f"status_gpu{gpu}.json"
        self.events_path = Path(log_dir) / f"events_gpu{gpu}.jsonl"
        self.gpu = gpu

    def status(self, **fields):
        write_json(self.status_path, fields)

    def event(self, message, **fields):
        with open(self.events_path, "a") as handle:
            handle.write(json.dumps({"time": time.time(), "message": f"[GPU {self.gpu}] {message}", **fields}) + "\n")
        print(message, flush=True)


def holm(p_values):
    """Holm-Bonferroni adjusted p-values, same order as the input."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted

The libraries are imported from `src/`, and every architecture is instantiated on the meta device (no memory) to count parameters and estimate GFLOPs per volume. Expected output: one row per model.

In [ ]:
sys.path.insert(0, str(SRC))
import shared_lib
import volume_lib as lib

importlib.reload(lib)
importlib.reload(shared_lib)
from torch.utils.flop_counter import FlopCounterMode

profiles = []
for name in MODELS:
    with torch.device("meta"):
        model = lib.build_model(name, IMG_SIZE)
        model.eval()
        try:
            with torch.no_grad(), FlopCounterMode(display=False) as counter:
                model(torch.zeros(1, 4, IMG_SIZE, IMG_SIZE, IMG_SIZE))
            gflops = counter.get_total_flops() / 1e9
        except Exception as exc:  # an op without a meta kernel only loses the FLOP estimate
            log(f"{name}: FLOPs not estimated ({type(exc).__name__})")
            gflops = float("nan")
    profiles.append(
        {
            "model": name,
            "family": lib.RECIPES[name]["family"],
            "parameters_m": sum(p.numel() for p in model.parameters()) / 1e6,
            "gflops_per_volume": gflops,
            **{k: v for k, v in lib.RECIPES[name].items() if k != "family"},
        }
    )
    del model
PROFILES = pd.DataFrame(profiles).set_index("model")
save_table(PROFILES, "model_profiles", index=True)

## 2. Data

### Find the cases and their grades

Locates the dataset under `/kaggle/input` (old and new layouts), lists every case folder with its four sequences and its mask, and joins the tumour grade (HGG or LGG) from `name_mapping.csv`. Expected output: the number of cases with all five files and the grade counts.

In [ ]:
def find_dataset_root(slug):
    for candidate in [INPUT / slug, *INPUT.glob(f"*/{slug}"), *INPUT.glob(f"*/*/{slug}")]:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Dataset '{slug}' is not attached: add it with 'Add Input' in the notebook sidebar."
    )


TRAIN_ROOT = next(find_dataset_root(DATASET_SLUG).rglob("MICCAI_BraTS2020_TrainingData"))
mapping_path = next(TRAIN_ROOT.glob("name_mapping*.csv"), None)
grades = {}
if mapping_path is not None:
    mapping = pd.read_csv(mapping_path)
    grades = dict(zip(mapping["BraTS_2020_subject_ID"], mapping["Grade"]))
rows = []
for case_dir in sorted(TRAIN_ROOT.glob("BraTS20_Training_*")):
    files = lib.case_files(case_dir)
    rows.append(
        {
            "case": case_dir.name,
            **{key: str(files.get(key, "")) for key in (*lib.MODALITIES, "seg")},
            "grade": grades.get(case_dir.name, "unknown"),
        }
    )
CASES = pd.DataFrame(rows)
complete = CASES[[*lib.MODALITIES, "seg"]].ne("").all(axis=1)
if (~complete).any():
    log(f"Cases missing a file (excluded): {CASES.loc[~complete, 'case'].tolist()}")
CASES = CASES[complete].reset_index(drop=True)
CASES["cache_index"] = np.arange(len(CASES))
log(f"{len(CASES)} complete cases | grades: {CASES['grade'].value_counts().to_dict()}")

### Preprocess and cache

One parallel pass (all CPU cores) applies the paper's preprocessing to every case: foreground-only z-score per sequence, trilinear resampling of the images and nearest-neighbour resampling of the mask to `IMG_SIZE`³ (the web app does the same), stored as float16 images and a 3-bit region mask in a memory-mapped cache. Native shape, voxel spacing and reference region volumes are recorded for native-resolution evaluation. Expected output: a progress bar and the cache size.

In [ ]:
needed = len(CASES) * IMG_SIZE**3 * (4 * 2 + 1)
assert shutil.disk_usage(CACHE).free > 1.2 * needed, f"Cache needs {needed / 1e9:.1f} GB in {CACHE}"
IMAGES_PATH, LABELS_PATH = CACHE / "images.npy", CACHE / "labels.npy"
images_out = np.lib.format.open_memmap(
    IMAGES_PATH, mode="w+", dtype=np.float16, shape=(len(CASES), 4, IMG_SIZE, IMG_SIZE, IMG_SIZE)
)
labels_out = np.lib.format.open_memmap(
    LABELS_PATH, mode="w+", dtype=np.uint8, shape=(len(CASES), IMG_SIZE, IMG_SIZE, IMG_SIZE)
)
info_rows = []
start = time.perf_counter()
jobs = [
    ({key: row[key] for key in (*lib.MODALITIES, "seg")}, IMG_SIZE)
    for row in CASES.to_dict("records")
]
with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
    for index, info in enumerate(
        tqdm(pool.map(lib.preprocess_case, jobs), total=len(jobs), desc="Preprocessing cases")
    ):
        images_out[index], labels_out[index] = info["image"], info["label"]
        info_rows.append(
            {
                "shape": "x".join(map(str, info["shape"])),
                "spacing": "x".join(f"{s:g}" for s in info["spacing"]),
                **{f"volume_cm3_{r}": v for r, v in zip(REGIONS, info["volumes_cm3"])},
            }
        )
images_out.flush()
labels_out.flush()
del images_out, labels_out
CASES = pd.concat([CASES, pd.DataFrame(info_rows)], axis=1)
IMAGES = np.load(IMAGES_PATH, mmap_mode="r")
LABELS = np.load(LABELS_PATH, mmap_mode="r")
log(
    f"Cached {len(CASES)} cases in {time.perf_counter() - start:.0f}s ({(IMAGES.nbytes + LABELS.nbytes) / 1e9:.2f} GB)"
)
display(CASES[["shape", "spacing"]].value_counts().rename("cases").to_frame())

### Dataset overview

Region volumes by grade (how often ET is absent, which drives the ET results) and one case with its four sequences and three region masks. Expected output: a table and two figures.

In [ ]:
volume_stats = CASES.groupby("grade")[[f"volume_cm3_{r}" for r in REGIONS]].describe().T
save_table(volume_stats.round(2), "region_volumes_by_grade", index=True)
no_et = (
    CASES.assign(no_et=CASES["volume_cm3_ET"] == 0).groupby("grade")["no_et"].agg(["sum", "mean"])
)
save_table(
    no_et.rename(columns={"sum": "cases_without_ET", "mean": "share"}).round(3),
    "cases_without_et",
    index=True,
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, region in zip(axes, REGIONS):
    for grade, sub in CASES.groupby("grade"):
        ax.hist(sub[f"volume_cm3_{region}"], bins=30, alpha=0.6, label=f"{grade} ({len(sub)})")
    ax.set(title=f"{region} volume per case", xlabel="cm³", ylabel="Cases")
    ax.legend()
fig.tight_layout()
save_fig(fig, "01_region_volumes")

example = CASES.loc[CASES["volume_cm3_ET"].idxmax()]
volume = IMAGES[example["cache_index"]].astype(np.float32)
regions = lib.bits_to_regions(LABELS[example["cache_index"]])
z = int(np.argmax(regions[0].sum(axis=(0, 1))))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, modality in enumerate(lib.MODALITIES):
    axes[0, col].imshow(np.rot90(volume[col, :, :, z]), cmap="gray")
    axes[0, col].set_title(modality.upper())
for col, region in enumerate(REGIONS):
    axes[1, col].imshow(np.rot90(volume[0, :, :, z]), cmap="gray")
    axes[1, col].imshow(
        np.rot90(np.ma.masked_where(~regions[col][:, :, z], regions[col][:, :, z])),
        cmap="autumn",
        alpha=0.6,
    )
    axes[1, col].set_title(f"{region} on FLAIR")
axes[1, 3].imshow(np.rot90(LABELS[example["cache_index"]][:, :, z]), cmap="viridis", vmin=0, vmax=7)
axes[1, 3].set_title("Region bits (WT=1, TC=2, ET=4)")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle(
    f"{example['case']} ({example['grade']}), axial slice {z} of the {IMG_SIZE}³ model input"
)
fig.tight_layout()
save_fig(fig, "02_modalities_and_masks")

### Grade-stratified split

About 70/15/15 train/validation/test, stratified by tumour grade, with a fixed seed so every model and seed sees the same cases. Expected output: cases per split and grade.

In [ ]:
strata = CASES["grade"]
train_val, test = train_test_split(
    CASES.index, test_size=TEST_FRACTION, stratify=strata, random_state=SPLIT_SEED
)
train, val = train_test_split(
    train_val,
    test_size=VAL_FRACTION / (1 - TEST_FRACTION),
    stratify=strata[train_val],
    random_state=SPLIT_SEED,
)
CASES["role"] = "train"
CASES.loc[val, "role"] = "val"
CASES.loc[test, "role"] = "test"
CASE_TABLE = METRICS / "cases.csv"
CASES.to_csv(CASE_TABLE, index=False)
split_table = pd.crosstab(
    CASES["role"], CASES["grade"], margins=True, margins_name="Total"
).reindex(["train", "val", "test", "Total"])
save_table(split_table, "split", index=True)
TEST_CASES = CASES[CASES["role"] == "test"].reset_index(drop=True)

### Augmentation preview

The GPU augmentation used in training (random flips on each axis, per-sequence intensity scale and shift), applied to one training case. Expected output: one figure.

In [ ]:
sample = CASES[CASES["role"] == "train"].iloc[0]
image = torch.from_numpy(IMAGES[sample["cache_index"]].astype(np.float32))[None]
label = torch.from_numpy(lib.bits_to_regions(LABELS[sample["cache_index"]]).astype(np.float32))[
    None
]
torch.manual_seed(SPLIT_SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    aug_image, aug_label = (image, label) if col == 0 else lib.augment(image, label)
    zc = IMG_SIZE // 2
    axes[0, col].imshow(np.rot90(aug_image[0, 0, :, :, zc].numpy()), cmap="gray")
    axes[1, col].imshow(np.rot90(aug_label[0, 0, :, :, zc].numpy()), cmap="gray")
    axes[0, col].set_title("Original FLAIR" if col == 0 else f"Augmented {col}")
    axes[1, col].set_title("WT mask")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "03_augmentation")

## 3. Training

Builds the job list (every model x seed, most expensive first so the two GPUs finish together), restores earlier runs from `RESUME_DIR` (after checking the split is identical), writes the worker configuration and launches one worker per GPU. The cell follows them live: an overall jobs bar, a step bar per GPU with loss, learning rate and peak memory, and one line per finished epoch with validation Dice per region. Jobs still unfinished at `TRAINING_BUDGET_HOURS` stop cleanly at an epoch boundary. Expected output: live progress.

In [ ]:
JOBS = [{"model": m, "seed": s} for m in MODELS for s in SEEDS]
JOBS.sort(key=lambda job: -PROFILES.loc[job["model"], "parameters_m"])
if RESUME_DIR is not None:
    previous = pd.read_csv(Path(RESUME_DIR) / "metrics" / "cases.csv")
    assert previous[["case", "role"]].equals(
        CASES[["case", "role"]]
    ), "RESUME_DIR was produced with a different split."
    for sub in ("metrics/runs", "predictions", "weights"):
        if (Path(RESUME_DIR) / sub).exists():
            shutil.copytree(Path(RESUME_DIR) / sub, WORK / sub, dirs_exist_ok=True)
    log(
        f"Restored {len(list(RUNS.glob('*/summary.json')))} finished runs and {len(list(WEIGHTS.glob('*/*_last.pth')))} resumable runs"
    )

shutil.rmtree(CLAIMS, ignore_errors=True)
CLAIMS.mkdir(parents=True)
for path in LOGS.glob("status_gpu*.json"):
    path.unlink()
worker_config = {
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "effective_batch": EFFECTIVE_BATCH,
    "grad_clip": GRAD_CLIP,
    "deterministic": DETERMINISTIC,
    "tta": TTA,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "et_candidates": ET_CANDIDATES,
    "loader_workers": LOADER_WORKERS,
    "latency_iters": LATENCY_ITERS,
    "images_path": str(IMAGES_PATH),
    "labels_path": str(LABELS_PATH),
    "case_table": str(CASE_TABLE),
    "runs_dir": str(RUNS),
    "weights_dir": str(WEIGHTS),
    "probs_dir": str(PREDICTIONS),
    "log_dir": str(LOGS),
    "claims_dir": str(CLAIMS),
    "deadline": time.time() + TRAINING_BUDGET_HOURS * 3600,
    "jobs": JOBS,
}
CONFIG_PATH = METRICS / "worker_config.json"
shared_lib.write_json(CONFIG_PATH, worker_config)


def finished_jobs():
    return sum((RUNS / lib.run_id(j["model"], j["seed"]) / "summary.json").exists() for j in JOBS)


log(f"{len(JOBS)} jobs, {finished_jobs()} already finished; launching {GPU_COUNT} workers")
training_start = time.perf_counter()
workers = shared_lib.launch_workers(SRC / "volume_worker.py", CONFIG_PATH, LOGS, GPU_COUNT)
shared_lib.monitor_workers(workers, LOGS, len(JOBS), finished_jobs)
TRAINING_WALL_SECONDS = time.perf_counter() - training_start
pending = [
    lib.run_id(j["model"], j["seed"])
    for j in JOBS
    if not (RUNS / lib.run_id(j["model"], j["seed"]) / "summary.json").exists()
]
log(
    f"Training stage finished in {TRAINING_WALL_SECONDS / 60:.1f} min; pending runs: {pending or 'none'}"
)

## 4. Results

### Training curves

Training and validation loss, and validation Dice per region, for every run that has a history (finished or paused). Expected output: one figure with a row per model.

In [ ]:
SUMMARIES = pd.DataFrame([shared_lib.read_json(p) for p in sorted(RUNS.glob("*/summary.json"))])
assert len(SUMMARIES), "No finished runs yet: rerun with RESUME_DIR after the next session."
models_done = [m for m in MODELS if m in set(SUMMARIES["model"])]
HISTORIES = {path.parent.name: pd.read_csv(path) for path in RUNS.glob("*/history.csv")}
models_with_history = [
    m for m in MODELS if any(k.startswith(lib.slug(m) + "_seed") for k in HISTORIES)
]
fig, axes = plt.subplots(
    len(models_with_history), 2, figsize=(15, 3 * len(models_with_history)), squeeze=False
)
for row, name in enumerate(models_with_history):
    for rid, history in HISTORIES.items():
        if not rid.startswith(lib.slug(name) + "_seed"):
            continue
        axes[row, 0].plot(history["epoch"], history["train_loss"], label=f"train ({rid})")
        axes[row, 0].plot(
            history["epoch"], history["val_loss"], linestyle="--", label=f"val ({rid})"
        )
        for region in REGIONS:
            axes[row, 1].plot(
                history["epoch"],
                history[f"val_dice_{region}"],
                label=f"{region} ({rid.split('_')[-1]})",
            )
        axes[row, 1].plot(
            history["epoch"], history["val_dice_mean"], color="black", linewidth=2, label="mean"
        )
    axes[row, 0].set_ylabel(name)
    axes[row, 0].legend(fontsize=7)
    axes[row, 1].legend(fontsize=7, ncol=2)
axes[0, 0].set_title("Loss")
axes[0, 1].set_title("Validation Dice (model grid, no TTA)")
fig.tight_layout()
save_fig(fig, "04_training_curves")

### Test results at native resolution

Per-case metrics of every finished run, averaged over seeds per case, then summarised per model: mean ± sd over cases with a bootstrap 95% CI of the mean Dice, plus HD95, sensitivity and specificity per region. With several seeds the seed-to-seed spread of the mean Dice is reported too. Expected output: the results table.

In [ ]:
case_frames = []
for summary in SUMMARIES.to_dict("records"):
    frame = pd.read_csv(RUNS / summary["run_id"] / "test_cases.csv")
    case_frames.append(frame.assign(model=summary["model"], seed=summary["seed"]))
PER_CASE_RUNS = pd.concat(case_frames, ignore_index=True)
metric_cols = [
    c
    for c in PER_CASE_RUNS.columns
    if c.startswith(("dice_", "hd95_", "sensitivity_", "specificity_"))
]
PER_CASE = PER_CASE_RUNS.groupby(["model", "case", "grade"], as_index=False)[
    metric_cols + ["tta_std_in_wt"]
].mean()
PER_CASE.to_csv(METRICS / "test_per_case.csv", index=False)

result_rows = []
for name in models_done:
    sub = PER_CASE[PER_CASE["model"] == name]
    low, high = lib.bootstrap_mean_ci(sub["dice_mean"], BOOTSTRAP_SAMPLES, SPLIT_SEED)
    seed_means = PER_CASE_RUNS[PER_CASE_RUNS["model"] == name].groupby("seed")["dice_mean"].mean()
    row = {
        "model": name,
        "family": lib.RECIPES[name]["family"],
        "seeds": len(seed_means),
        "cases": len(sub),
        "dice_mean": sub["dice_mean"].mean(),
        "dice_mean_ci": f"[{low:.4f}, {high:.4f}]",
        "dice_mean_seed_sd": seed_means.std() if len(seed_means) > 1 else np.nan,
    }
    for region in REGIONS:
        row[f"dice_{region}"] = sub[f"dice_{region}"].mean()
        row[f"dice_{region}_sd"] = sub[f"dice_{region}"].std()
        row[f"hd95_{region}"] = sub[f"hd95_{region}"].median()
        row[f"sensitivity_{region}"] = sub[f"sensitivity_{region}"].mean()
        row[f"specificity_{region}"] = sub[f"specificity_{region}"].mean()
    result_rows.append(row)
RESULTS = pd.DataFrame(result_rows).set_index("model").sort_values("dice_mean", ascending=False)
save_table(RESULTS.round(4), "test_results", index=True)
log(
    "HD95 is summarised by its median because empty-region cases contribute the fixed BraTS value of 373.13 mm."
)

### Ensembles

Seed ensembles per model (mean probability over seeds), an ensemble of all finished models and one of the CNN models only, evaluated with the same native-resolution metrics. Each ensemble uses the median of its members' ET size thresholds. HD95 runs on all CPU cores. Expected output: the ensemble rows added to the results.

In [ ]:
def ensemble_probs(names, case_position):
    members = []
    for name in names:
        for summary in SUMMARIES[SUMMARIES["model"] == name].to_dict("records"):
            probs = np.load(PREDICTIONS / f"{summary['run_id']}.npy", mmap_mode="r")
            members.append(probs[case_position].astype(np.float32) / 255)
    return np.mean(members, axis=0)


def score_case(args):
    pred, seg_path = args
    reference, spacing = lib.load_reference(seg_path)
    return lib.case_metrics(pred, reference, spacing)


ENSEMBLES = {
    f"{m} (seed ensemble)": [m] for m in models_done if (SUMMARIES["model"] == m).sum() > 1
}
ENSEMBLES["All models"] = models_done
cnn_models = [m for m in models_done if lib.RECIPES[m]["family"] == "CNN"]
if 1 < len(cnn_models) < len(models_done):
    ENSEMBLES["CNN models"] = cnn_models
ensemble_rows = []
for label_e, names in ENSEMBLES.items():
    et_min = int(np.median(SUMMARIES[SUMMARIES["model"].isin(names)]["et_min_voxels"]))
    tasks = []
    for position, case in enumerate(
        tqdm(TEST_CASES.itertuples(), total=len(TEST_CASES), desc=label_e)
    ):
        shape = tuple(int(v) for v in case.shape.split("x"))
        probs = torch.from_numpy(ensemble_probs(names, position)).to(DEVICE)
        tasks.append(
            (lib.regions_from_probs(lib.to_native(probs, shape), et_min).cpu().numpy(), case.seg)
        )
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
        metrics_e = list(pool.map(score_case, tasks))
    frame_e = pd.DataFrame(metrics_e).assign(
        model=label_e, case=TEST_CASES["case"].values, grade=TEST_CASES["grade"].values
    )
    ensemble_rows.append(frame_e)
    del tasks
ENSEMBLE_CASES = pd.concat(ensemble_rows, ignore_index=True)
ENSEMBLE_CASES.to_csv(METRICS / "ensemble_per_case.csv", index=False)
ens_summary = ENSEMBLE_CASES.groupby("model").agg(
    dice_mean=("dice_mean", "mean"),
    **{f"dice_{r}": (f"dice_{r}", "mean") for r in REGIONS},
    **{f"hd95_{r}": (f"hd95_{r}", "median") for r in REGIONS},
)
ens_summary["dice_mean_ci"] = [
    "[{:.4f}, {:.4f}]".format(
        *lib.bootstrap_mean_ci(
            ENSEMBLE_CASES.loc[ENSEMBLE_CASES["model"] == m, "dice_mean"],
            BOOTSTRAP_SAMPLES,
            SPLIT_SEED,
        )
    )
    for m in ens_summary.index
]
save_table(ens_summary.round(4), "ensemble_results", index=True)

### Statistical comparison

Pairwise Wilcoxon signed-rank tests on per-case mean Dice (paired by case, seed-averaged), Holm-corrected, and the median paired difference. Expected output: a table and a p-value heatmap.

In [ ]:
wide = PER_CASE.pivot(index="case", columns="model", values="dice_mean")[models_done]
pairs = [(a, b) for i, a in enumerate(models_done) for b in models_done[i + 1 :]]
raw = []
for a, b in pairs:
    diff = wide[a] - wide[b]
    raw.append(1.0 if np.allclose(diff, 0) else wilcoxon(wide[a], wide[b]).pvalue)
adjusted = shared_lib.holm(raw) if raw else []
P_MATRIX = pd.DataFrame(1.0, index=models_done, columns=models_done)
test_rows = []
for (a, b), p_raw, p_adj in zip(pairs, raw, adjusted):
    P_MATRIX.loc[a, b] = P_MATRIX.loc[b, a] = p_adj
    test_rows.append(
        {
            "model_a": a,
            "model_b": b,
            "median_dice_difference": float((wide[a] - wide[b]).median()),
            "p_raw": p_raw,
            "p_holm": p_adj,
        }
    )
save_table(pd.DataFrame(test_rows).round(5), "wilcoxon_tests")
if len(models_done) > 1:
    fig, ax = plt.subplots(figsize=(1.2 * len(models_done) + 3, 1.0 * len(models_done) + 2))
    image = ax.imshow(-np.log10(P_MATRIX.to_numpy().clip(1e-12)), cmap="viridis")
    ax.set_xticks(range(len(models_done)), models_done, rotation=30, ha="right")
    ax.set_yticks(range(len(models_done)), models_done)
    for i in range(len(models_done)):
        for j in range(len(models_done)):
            ax.text(
                j,
                i,
                f"{P_MATRIX.iat[i, j]:.3g}",
                ha="center",
                va="center",
                color="white",
                fontsize=8,
            )
    ax.grid(False)
    plt.colorbar(image, ax=ax, label="-log10 Holm-adjusted p")
    ax.set_title("Wilcoxon signed-rank tests on per-case mean Dice")
    fig.tight_layout()
    save_fig(fig, "05_wilcoxon")

### Results by tumour grade

Mean Dice per region for HGG and LGG cases separately, with bootstrap CIs of the mean Dice, to check whether models trained on a mostly HGG cohort underperform on LGG. Expected output: a table and a figure.

In [ ]:
grade_rows = []
for (name, grade), sub in PER_CASE.groupby(["model", "grade"]):
    low, high = lib.bootstrap_mean_ci(sub["dice_mean"], BOOTSTRAP_SAMPLES, SPLIT_SEED)
    grade_rows.append(
        {
            "model": name,
            "grade": grade,
            "cases": len(sub),
            **{f"dice_{r}": sub[f"dice_{r}"].mean() for r in REGIONS},
            "dice_mean": sub["dice_mean"].mean(),
            "ci_low": low,
            "ci_high": high,
        }
    )
BY_GRADE = pd.DataFrame(grade_rows)
save_table(BY_GRADE.round(4), "test_results_by_grade")
fig, ax = plt.subplots(figsize=(12, 4.5))
grades_present = sorted(BY_GRADE["grade"].unique())
for k, grade in enumerate(grades_present):
    sub = BY_GRADE[BY_GRADE["grade"] == grade].set_index("model").reindex(models_done)
    x = np.arange(len(models_done)) + (k - (len(grades_present) - 1) / 2) * 0.35
    ax.bar(
        x,
        sub["dice_mean"],
        0.35,
        yerr=[sub["dice_mean"] - sub["ci_low"], sub["ci_high"] - sub["dice_mean"]],
        capsize=3,
        label=f"{grade} ({int(sub['cases'].max())} cases)",
    )
ax.set_xticks(range(len(models_done)), models_done)
ax.set(title="Test mean Dice by tumour grade (95% bootstrap CI)", ylabel="Mean Dice", ylim=(0, 1))
ax.legend()
save_fig(fig, "06_dice_by_grade")

### Per-case distributions

Box plots of per-case Dice and HD95 for every region and model, including the ensembles. Expected output: one figure.

In [ ]:
combined = pd.concat([PER_CASE, ENSEMBLE_CASES], ignore_index=True)
order = models_done + [m for m in ENSEMBLE_CASES["model"].unique() if m not in models_done]
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for col, region in enumerate(REGIONS):
    for row, metric in enumerate(("dice", "hd95")):
        data = [combined.loc[combined["model"] == m, f"{metric}_{region}"].dropna() for m in order]
        axes[row, col].boxplot(data, showfliers=True)
        axes[row, col].set_xticks(
            range(1, len(order) + 1), order, rotation=35, ha="right", fontsize=8
        )
        axes[row, col].set_title(
            f"{region} {'Dice' if metric == 'dice' else 'HD95 (mm, log scale)'}"
        )
        if metric == "hd95":
            axes[row, col].set_yscale("log")
fig.tight_layout()
save_fig(fig, "07_per_case_distributions")

### Qualitative comparison

For the test cases with the median and the lowest all-model-ensemble Dice, the axial slice with the largest reference tumour, showing the reference and every model's prediction (seed ensemble) on FLAIR. Colours: oedema-only WT in green, TC in yellow, ET in red. Expected output: one figure.

In [ ]:
def overlay(ax, flair, regions, title):
    ax.imshow(np.rot90(flair), cmap="gray")
    colour = np.zeros((*flair.shape, 4))
    colour[regions[0]] = (0.0, 0.8, 0.0, 0.45)
    colour[regions[1]] = (1.0, 0.85, 0.0, 0.55)
    colour[regions[2]] = (1.0, 0.0, 0.0, 0.65)
    ax.imshow(np.rot90(colour, axes=(0, 1)))
    ax.set_title(title, fontsize=8)
    ax.axis("off")


all_scores = ENSEMBLE_CASES[ENSEMBLE_CASES["model"] == "All models"].reset_index(drop=True)
picks = {
    "median case": int((all_scores["dice_mean"] - all_scores["dice_mean"].median()).abs().idxmin()),
    "hardest case": int(all_scores["dice_mean"].idxmin()),
}
fig, axes = plt.subplots(
    len(picks),
    len(models_done) + 1,
    figsize=(2.8 * (len(models_done) + 1), 3.2 * len(picks)),
    squeeze=False,
)
for row, (label_p, position) in enumerate(picks.items()):
    case = TEST_CASES.iloc[position]
    reference = lib.bits_to_regions(LABELS[case["cache_index"]])
    z = int(np.argmax(reference[0].sum(axis=(0, 1))))
    flair = IMAGES[case["cache_index"]][0, :, :, z].astype(np.float32)
    overlay(
        axes[row, 0],
        flair,
        reference[:, :, :, z],
        f"Reference\n{case['case']} ({case['grade']}, {label_p})",
    )
    for col, name in enumerate(models_done, start=1):
        probs = torch.from_numpy(ensemble_probs([name], position))
        pred = lib.regions_from_probs(probs).numpy()
        score = PER_CASE[(PER_CASE["model"] == name) & (PER_CASE["case"] == case["case"])][
            "dice_mean"
        ].iloc[0]
        overlay(axes[row, col], flair, pred[:, :, :, z], f"{name}\nmean Dice {score:.3f}")
fig.tight_layout()
save_fig(fig, "08_qualitative_comparison")

### Uncertainty and error

With TTA on, each case has a mean voxel-wise standard deviation across the 8 flips inside its predicted whole tumour. A useful uncertainty signal should be higher for worse cases: the Spearman correlation with per-case mean Dice should be negative. Expected output: a table and a scatter figure, or a note that TTA is off.

In [ ]:
if not TTA:
    log("TTA is off: no uncertainty estimate.")
else:
    unc_rows = []
    fig, ax = plt.subplots(figsize=(9, 6))
    for name in models_done:
        sub = PER_CASE[PER_CASE["model"] == name]
        rho, p = spearmanr(sub["tta_std_in_wt"], sub["dice_mean"])
        unc_rows.append({"model": name, "spearman_rho": rho, "p_value": p})
        ax.scatter(
            sub["tta_std_in_wt"],
            sub["dice_mean"],
            s=14,
            alpha=0.6,
            color=MODEL_COLOURS[name],
            label=f"{name} (rho {rho:.2f})",
        )
    ax.set(
        title="Per-case TTA uncertainty vs mean Dice",
        xlabel="Mean flip standard deviation inside predicted WT",
        ylabel="Mean Dice",
    )
    ax.legend(fontsize=8)
    save_fig(fig, "09_uncertainty_vs_dice")
    save_table(pd.DataFrame(unc_rows).round(4), "uncertainty_correlation")

### Efficiency

Parameters, estimated GFLOPs per volume, training time and steps, peak training and inference memory, and latency per volume (batch 1, fp16, mean ± sd over timed passes after warm-up) without and with the 8-flip TTA, plus Dice per million parameters and per training hour. Expected output: a table and a figure.

In [ ]:
eff = SUMMARIES.groupby("model").agg(
    seeds=("seed", "count"),
    epochs_run=("epochs_run", "mean"),
    optimizer_steps=("optimizer_steps", "mean"),
    micro_batch=("micro_batch", "first"),
    train_hours=("train_seconds", lambda s: s.mean() / 3600),
    peak_train_memory_gb=("peak_train_memory_gb", "max"),
    peak_inference_memory_gb=("peak_inference_memory_gb", "max"),
    latency_s=("latency_s", "mean"),
    latency_s_sd=("latency_s_sd", "mean"),
    latency_tta_s=("latency_tta_s", "mean"),
    latency_tta_s_sd=("latency_tta_s_sd", "mean"),
)
EFFICIENCY = (
    PROFILES[["parameters_m", "gflops_per_volume"]]
    .join(eff, how="inner")
    .join(RESULTS[["dice_mean"]])
)
EFFICIENCY["dice_per_m_params"] = EFFICIENCY["dice_mean"] / EFFICIENCY["parameters_m"]
EFFICIENCY["dice_per_train_hour"] = EFFICIENCY["dice_mean"] / EFFICIENCY["train_hours"]
save_table(EFFICIENCY.round(4), "efficiency", index=True)
save_table(
    SUMMARIES[["run_id", "et_min_voxels", "best_epoch", "best_val_dice"]].round(4),
    "et_threshold_and_selection",
)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    EFFICIENCY["latency_s"],
    EFFICIENCY["dice_mean"],
    s=30 + 4 * EFFICIENCY["parameters_m"],
    alpha=0.6,
)
for name, row in EFFICIENCY.iterrows():
    ax.annotate(
        f"{name}\n{row['parameters_m']:.1f}M, {row['train_hours']:.1f} h",
        (row["latency_s"], row["dice_mean"]),
        fontsize=8,
        xytext=(5, 5),
        textcoords="offset points",
    )
ax.set(
    title="Test mean Dice vs latency per volume (bubble area: parameters)",
    xlabel="Latency (s per volume, no TTA)",
    ylabel="Mean Dice",
)
save_fig(fig, "10_dice_vs_latency")

## 5. Export and summary

### Web-app checkpoint

The web app loads `VNet.pth` into its `VNet3D` class (`Backend/neuroscan/inference/networks.py`), resamples each sequence to 128³ after the same foreground z-score, and reads the first output. The VNet run with the best validation Dice is exported, loaded strictly into this notebook's `VNet3D` (identical module names) and checked against its stored test probabilities. Copy `weights/deploy/VNet.pth` to `Backend/Segmentation/` to deploy it. Expected output: the export path and a passed check.

In [ ]:
DEPLOY = WEIGHTS / "deploy"
DEPLOY.mkdir(exist_ok=True)
deploy_runs = SUMMARIES[SUMMARIES["model"] == DEPLOY_MODEL]
if deploy_runs.empty or IMG_SIZE != 128:
    log(
        f"No export: {DEPLOY_MODEL} has no finished run, or IMG_SIZE differs from the web app's 128"
    )
else:
    run = deploy_runs.loc[deploy_runs["best_val_dice"].idxmax()]
    target = DEPLOY / f"{DEPLOY_MODEL}.pth"
    shutil.copyfile(WEIGHTS / lib.slug(DEPLOY_MODEL) / f"{run['run_id']}.pth", target)
    model = lib.VNet3D()
    model.load_state_dict(torch.load(target, map_location="cpu", weights_only=True), strict=True)
    model = model.to(DEVICE).eval()
    case = TEST_CASES.iloc[0]
    x = torch.from_numpy(IMAGES[case["cache_index"]].astype(np.float32))[None].to(DEVICE)
    probs, _ = lib.predict_probs(model, x, TTA)
    stored = (
        np.load(PREDICTIONS / f"{run['run_id']}.npy", mmap_mode="r")[0].astype(np.float32) / 255
    )
    max_diff = float(np.abs(probs[0].cpu().numpy() - stored).max())
    assert (
        max_diff < 0.02
    ), f"Exported checkpoint does not reproduce the stored probabilities (max diff {max_diff:.4f})"
    metadata = {
        "architecture": "VNet3D (Backend/neuroscan/inference/networks.py)",
        "input": "FLAIR, T1, T1CE, T2; foreground z-score; trilinear resample to 128^3",
        "output": "WT, TC, ET logits (first element of the output tuple)",
        "run_id": run["run_id"],
        "best_val_dice": float(run["best_val_dice"]),
        "test_dice_mean": float(run["test_dice_mean"]),
        "et_min_voxels_native": int(run["et_min_voxels"]),
    }
    (DEPLOY / "metadata.json").write_text(json.dumps(metadata, indent=2))
    log(
        f"Exported {target} ({target.stat().st_size / 1e6:.1f} MB); reload check passed (max diff {max_diff:.4f})"
    )
    del model

### Summary

The headline table, the run environment, the runs still pending, and the limitations that remain. Expected output: the summary table and `metrics/run_summary.json`.

In [ ]:
summary_table = RESULTS[
    [
        "family",
        "seeds",
        "dice_mean",
        "dice_mean_ci",
        "dice_WT",
        "dice_TC",
        "dice_ET",
        "hd95_WT",
        "hd95_TC",
        "hd95_ET",
    ]
].join(EFFICIENCY[["parameters_m", "train_hours", "latency_s", "peak_train_memory_gb"]])
save_table(summary_table.round(4), "summary_table", index=True)
run_summary = {
    "cases": len(CASES),
    "split": CASES["role"].value_counts().to_dict(),
    "epochs": EPOCHS,
    "seeds": SEEDS,
    "tta": TTA,
    "deterministic": DETERMINISTIC,
    "finished_runs": SUMMARIES["run_id"].tolist(),
    "pending_runs": pending,
    "best_model": RESULTS.index[0],
    "ensembles": ens_summary["dice_mean"].round(4).to_dict(),
    "training_wall_minutes": TRAINING_WALL_SECONDS / 60,
    "notebook_minutes": (time.time() - NOTEBOOK_START) / 60,
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpus": [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)],
    },
    "limitations": [
        "single dataset (BraTS 2020 training cases); no external multi-centre test set with public labels",
        "no large-scale pretraining of the transformer encoders",
        f"one input resolution per run ({IMG_SIZE}^3, whole volume resampled)",
    ],
}
(METRICS / "run_summary.json").write_text(json.dumps(run_summary, indent=2, default=float))
log(
    f"Best model: {run_summary['best_model']}; finished {len(run_summary['finished_runs'])} runs, pending {len(pending)}"
)

### Package the outputs

Zips everything in `/kaggle/working/` (the volume cache lives in `/tmp` and is not included), storing already-compressed files without recompression, and shows a download link. The zip also appears in the notebook's **Output** tab after a saved run. Expected output: the file count and a download link.

In [ ]:
ZIP_PATH = WORK / "outputs.zip"
EXCLUDE_PARTS = {"__pycache__", ".cache", ".ipynb_checkpoints", "cache", "tmp", "wandb"}
STORED_SUFFIXES = {
    ".zip",
    ".png",
    ".jpg",
    ".jpeg",
    ".pt",
    ".pth",
    ".bin",
    ".safetensors",
    ".gz",
    ".npz",
    ".npy",
}
files = [
    path
    for path in WORK.rglob("*")
    if path.is_file()
    and path != ZIP_PATH
    and not EXCLUDE_PARTS.intersection(path.relative_to(WORK).parts)
]
total_bytes = sum(path.stat().st_size for path in files)
free_bytes = shutil.disk_usage(WORK).free
assert (
    free_bytes > total_bytes * 1.05
), f"Not enough disk space to zip {total_bytes / 1e9:.2f} GB with {free_bytes / 1e9:.2f} GB free."
with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as archive:
    for path in tqdm(files, desc="Zipping outputs", unit="file"):
        method = (
            zipfile.ZIP_STORED if path.suffix.lower() in STORED_SUFFIXES else zipfile.ZIP_DEFLATED
        )
        archive.write(path, path.relative_to(WORK), compress_type=method)
print(f"Zipped {len(files)} files ({total_bytes / 1e9:.2f} GB) into {ZIP_PATH}", flush=True)
display(FileLink(os.path.relpath(ZIP_PATH)))